# Phase 2 · 21단원 — 피처 스토어와 학습-서빙 괴리

> 특징은 **정의가 하나**인데 **읽는 쪽이 둘**입니다.
> 학습은 각 라벨 시점의 **과거 값**을 읽고, 서빙은 **가장 최근 값**을 읽죠.
> 둘이 어긋나면 모델은 **학습한 적 없는 입력**을 받게 됩니다.

- 원문: <https://aiengineeringfromscratch.com/lesson?path=phases/02-ml-fundamentals/21-feature-stores-and-training-serving-skew>
- 예상 시간: 100분
- 먼저 보면 좋은 것: 8단원(특징 공학), 13단원(파이프라인), 15단원(시계열), 19단원(시스템 설계)
- 표시 규칙: 📗 는 원문에 있는 내용, ✨ 는 제가 넣은 예시·실험, ⚠️ 는 원문과 측정이 달랐던 곳입니다.

---

## 이 단원이 푸는 문제

어떤 팀이 이탈 모델을 만듭니다.

라벨은 **"기준일 이후 30일 동안 구매가 없었는가"** 입니다.
라벨을 사용자 특징 테이블에 `user_id` 로 조인하고,
`days_since_last_purchase` 로 규칙을 학습시킵니다.

**오프라인 정확도 0.948.** 훌륭하죠.

그런데 실제 트래픽에서는 **더 낡고 단순한 예전 버전보다 못합니다.**

### 왜 그랬을까요? 📗 원문이 두 가지를 짚습니다

**① 특징 테이블에 '가장 최신 값' 만 들어 있었습니다.**

라벨 기준일로부터 **몇 주 뒤**의 값이었죠.
115일에 마지막 구매를 한 사용자가 **175일 스냅샷에서는 60일 쉰 것**으로 나옵니다.
그런데 라벨이 묻는 게 바로 그겁니다. **"이 사람이 구매를 멈췄나?"**

특징 안에 **정답이 들어 있었던** 겁니다.

**② 서빙 쪽 특징을 다른 팀이 다른 서비스에서 또 짰습니다.**

- 30일 창에 **하루가 더** 들어가 있었고
- 신규 사용자 기본값이 **365 가 아니라 -1** 이었습니다

### 두 결함의 원인은 같습니다

> **학습과 서빙이 "같은 특징"을 서로 다른 방법으로 계산했습니다.**

## 그래서 이 단원이 만드는 것

피처 스토어의 부품들을 밑바닥부터 만듭니다. 전부 표준 라이브러리로요.

| 부품 | 하는 일 |
| --- | --- |
| **레지스트리** | 특징 정의를 **한 곳에** 두고 양쪽이 같이 읽습니다 |
| **백필** | 새 특징의 **과거 값**을 날짜별로 채웁니다 |
| **시점 정확 조인** | 각 라벨에 **그 시점에 존재했던** 값만 붙입니다 |
| **온라인 스토어 (TTL)** | 엔티티당 최신 값 하나. 너무 오래된 건 없는 셈 칩니다 |
| **괴리 검사** | 학습 경로와 서빙 경로가 **진짜 같은 값**을 내는지 확인 |

그리고 **위의 두 결함을 숫자로 재현합니다.**

---

## 이걸 배우면 뭘 할 수 있나

> **"오프라인에선 잘 나왔는데 실서비스에서 안 나와요."**

이 말을 들었을 때 **어디를 봐야 할지** 알게 됩니다.

| 질문 | 어디서 | 이 단원의 답 |
| --- | --- | --- |
| 학습 데이터에 **미래가 섞였나?** | 3절 | 시점 정확 조인으로 바꿔 보면 바로 드러납니다 |
| 학습과 서빙이 **같은 코드**를 쓰나? | 5절 | 정의를 레지스트리 한 곳에 둡니다 |
| 서빙이 읽는 값이 **얼마나 묵었나?** | 7절 | TTL 과 머티리얼라이제이션 |
| 둘이 **진짜 같은 값**을 내나? | 8절 | 세 쌍을 비교하면 원인까지 좁혀집니다 |

그리고 ⚠️ 하나. **괴리 수치가 크다고 성능이 나쁜 건 아닙니다**(9절).
괴리 검사는 **"두 쪽이 다르다"** 를 알려 줄 뿐 **"어느 쪽이 맞다"** 는 안 알려 줍니다.
이걸 모르면 멀쩡한 경로를 고치느라 시간을 씁니다.

---

## 이 단원의 줄거리

20단원은 **세상이 변하는** 문제였습니다. 21단원은 **세상은 그대로인데 우리 코드가 두 벌이라** 생기는 문제입니다.

**1절에서 왜 코드가 두 벌이 되는지** 봅니다.
학습과 서빙은 **묻는 질문 자체가 다릅니다.**
"천만 개 (대상, 시점) 쌍의 값은?" 과 "이 사람 지금 값은?" 이요.
그래서 저장소도 둘로 나뉩니다.

**2절은 용어**입니다. 엔티티, 이벤트 시각, 생성 시각, TTL.
짧지만 3절을 이해하려면 꼭 필요합니다. 특히 **이벤트 시각**이요.

**3절이 이 단원에서 가장 중요합니다.** 시점 정확 조인입니다.
라벨 하나하나에 **그 시점에 존재했던 값만** 붙이는 거예요.
안 그러면 어떻게 되는지 숫자로 봅니다. **오프라인 0.948, 실제 0.763.**
누수된 쪽이 **더 좋아 보이면서 더 나쁩니다.**

**4절에서 괴리의 원인 네 가지**를 정리합니다. 3절이 그중 하나(시간 여행)고요.

**5~7절에서 부품을 만듭니다.** 레지스트리 · 백필 · 온라인 스토어.
"정의를 한 곳에 둔다"가 어떻게 코드가 되는지 봅니다.

**8절에서 괴리를 탐지합니다.** 세 쌍을 비교하면 **원인까지** 좁혀집니다.
어느 쌍이 깨졌느냐가 범인을 가리킵니다.

**9절이 ⚠️ 입니다.** 8절에서 괴리율 78.6% 가 나오는데,
그게 **정확도를 떨어뜨리는지** 재 봅니다. 결과가 예상과 다릅니다.

**10절**에서 TTL 과 스냅샷 간격을 바꿔 가며 비용과 신선도의 맞교환을 봅니다.

> 한 줄로 줄이면 이렇습니다.
> **정의를 한 곳에 두고, 학습은 그 시점 값만 읽고,
> 서빙은 같은 정의로 계산하고, 그 둘이 같은지 계속 확인한다.**

<details>
<summary>절 목록 (펼쳐 보기)</summary>

| 절 | 내용 | 출처 |
| --- | --- | --- |
| 1 | 같은 특징, 두 명의 독자 | 📗 원문 |
| 2 | 엔티티 · 이벤트 시각 · TTL | 📗 원문 |
| 3 | 시점 정확 조인 — 미래가 새는 걸 막기 | 📗 원문 + ✨ 측정 |
| 4 | 학습-서빙 괴리의 네 가지 원인 | 📗 원문 |
| 5 | 특징 레지스트리 — 정의는 하나 | 📗 원문 |
| 6 | 백필 — 과거를 채우기 | 📗 원문 |
| 7 | 온라인 스토어와 TTL | 📗 원문 |
| 8 | 괴리 탐지 — 세 쌍을 비교하면 범인이 좁혀집니다 | 📗 원문 |
| 9 | ⚠️ 괴리가 크다고 성능이 나쁜 건 아닙니다 | 📗 원문 + ✨ 반증 |
| 10 | ⚠️ TTL 과 스냅샷 간격 | 📗 연습문제 + ✨ 측정 |
| 11 | 정리 · Feast · Ship It · 용어 · 퀴즈 | 📗 원문 |

</details>

---

## 0. 준비

In [1]:
import bisect
import math
import random
import warnings
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.font_manager import findfont, FontProperties

warnings.filterwarnings("ignore")

후보들 = ["AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic"]
찾은것 = None
for 후보 in 후보들:
    try:
        if 후보.lower().replace(" ", "") in findfont(FontProperties(family=후보)).lower().replace(" ", ""):
            찾은것 = 후보
            break
    except Exception:
        pass
matplotlib.rcParams["font.family"] = ["DejaVu Sans"] + ([찾은것] if 찾은것 else [])
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["figure.dpi"] = 110

print("준비 끝!  한글 폰트:", 찾은것 or "(없음)")

준비 끝!  한글 폰트: AppleGothic


---

## 1. 같은 특징, 두 명의 독자

<sub>📗 원문 "Two readers of the same feature"</sub>

학습과 서빙은 특징을 **다른 모양으로** 요구합니다.

| 성질 | 학습 (오프라인) | 서빙 (온라인) |
| --- | --- | --- |
| **묻는 질문** | "천만 개 **(대상, 시점)** 쌍 각각의 값은?" | "**이 대상**의 **지금** 값은?" |
| 시점 | 과거의 **여러 지점** | **현재** |
| 양 | 작업당 수백만 행 | 요청당 한 행 |
| 지연 | 몇 분 걸려도 됨 | **몇 밀리초** |
| 저장소 | **오프라인 스토어** — 타임스탬프가 붙은 전체 이력 | **온라인 스토어** — 대상당 최신 값 하나, 키-값 DB |

### 🔑 질문이 다르니 저장소도 둘입니다

왼쪽은 **이력 전체**가 필요하고 느려도 됩니다.
오른쪽은 **최신 값 하나**만 필요하고 빨라야 합니다.

같은 데이터를 한 저장소로 둘 다 감당하려면 어느 한쪽이 희생됩니다.
그래서 **둘로 나누고**, **양쪽 다 같은 정의로** 채웁니다.

> 📗 **머티리얼라이제이션**(materialization)은 오프라인 스토어의 최신 값을
> 온라인 스토어로 **복사하는 작업**입니다.

### 흐름도

```
원본 이벤트
    ↓
특징 정의 (레지스트리)        ← 정의는 여기 한 곳에만
    ↓
오프라인 스토어 (타임스탬프 붙은 이력)
    ↓                    ↓
시점 정확 조인          머티리얼라이즈
    ↓                    ↓
학습 데이터            온라인 스토어 (대상당 최신 값)
    ↓                    ↓
    │                 서빙 요청
    │                    ↓
    │                 기록된 특징
    └──────→ 괴리 검사 ←──┘
```

맨 아랫줄을 보세요. **학습 경로와 서빙 경로가 다시 만나서** 서로 같은지 확인합니다.
그게 8절입니다.

> **🧩 이 절의 결론**
>
> 학습과 서빙은 **묻는 질문이 다릅니다.** 그래서 저장소를 둘로 나눕니다.
> 대신 **정의는 한 곳**에 두고 양쪽이 같이 읽습니다. 그게 피처 스토어의 핵심입니다.
>
> **➡️ 다음**
>
> 저장소를 나눴으니 **각 행이 언제 것인지** 적어 둬야 합니다.
> 그 용어들이 2절입니다. 짧지만 3절을 이해하려면 꼭 필요합니다.

---

## 2. 엔티티 · 이벤트 시각 · TTL

<sub>📗 원문 "Entities, timestamps, and TTL"</sub>

특징 행 하나를 설명하는 **네 가지 용어**입니다.

| 용어 | 뜻 | 예 |
| --- | --- | --- |
| **엔티티** | 그 특징이 **설명하는 대상의 키** | `user_id`, `card_id` |
| **이벤트 시각** | 그 값이 **참이 된 시점** | "14일 기준 구매 수"의 이벤트 시각은 14일 |
| **생성 시각** | 시스템이 그 행을 **쓴 시점** | 오늘 돌린 백필은 **옛 이벤트 시각 + 오늘 생성 시각** |
| **TTL** | 값의 **최대 나이**. 넘으면 없는 셈 치고 기본값을 씁니다 | 10일 |

### 🔑 이벤트 시각과 생성 시각이 왜 다를까요

**백필** 때문입니다(6절).

새 특징을 만들면 과거 값이 없습니다. 그래서 오늘 **과거 날짜마다 다시 계산**하죠.
그러면 그 행의 **이벤트 시각은 과거**지만 **생성 시각은 오늘**입니다.

> 📗 여기에 **미묘한 누수**가 있습니다.
> 오늘 돌린 백필은 **늦게 도착했거나 나중에 수정된 데이터**를 읽을 수 있습니다.
> 엄격한 시점 정확 조인은 **생성 시각도 라벨 시점 이전인지** 확인합니다.

같은 엔티티 · 같은 원본 · 같은 TTL 을 공유하는 특징 묶음을
**피처 뷰**(feature view)라고 부릅니다.

> **🧩 이 절의 결론**
>
> 특징 행마다 **엔티티 · 이벤트 시각 · 생성 시각 · TTL** 이 붙습니다.
> **이벤트 시각**(값이 참이 된 때)과 **생성 시각**(행을 쓴 때)은 다를 수 있습니다. 백필이 그렇죠.
>
> **➡️ 다음**
>
> 이제 가장 중요한 절입니다. 라벨에 특징을 붙이는 법이요.
> **틀리게 붙이면 정답이 입력으로 새어 들어갑니다.** 3절입니다.

---

## 3. 시점 정확 조인 — 미래가 새는 걸 막기

<sub>📗 원문 "Point-in-time correct joins"</sub>

학습 행 하나는 **라벨**과 **그 라벨 시점에 존재했던 특징 값**을 짝지은 것입니다.

### 📗 시점 정확 조인의 네 단계

1. **같은 엔티티**의 특징 행들을 찾습니다
2. **이벤트 시각이 라벨 시각 이하**인 행만 남깁니다
3. 그중 **가장 최근 것**을 고릅니다
4. 라벨 시각 − 이벤트 시각이 **TTL 보다 크면 버립니다**

### 📗 한 사용자, 16일 라벨, 주간 특징 행

| 특징 행 날짜 | `purchases_30d` | 시점 정확 조인 | 순진한 조인 (`user_id` 로만) |
| --- | --- | --- | --- |
| 7 | 4 | | |
| 14 | 3 | ✅ **선택** | |
| 21 | 1 | ❌ 미래 | |
| 28 | 0 | ❌ 미래 | ✅ **선택** |

### 🔑 맨 아랫줄이 왜 치명적인가

라벨이 묻는 건 **"이 사용자가 17~46일에 뭔가 샀는가"** 입니다.

그런데 **28일 행은 이미 라벨 기간이 끝난 뒤**를 세고 있습니다.
구매가 0 인 이유가 **사용자가 구매를 멈췄기 때문**이고,
그게 **바로 라벨이 재려던 사건**이에요.

> **순진한 조인은 정답을 입력에 복사해 넣습니다.**

📗 원문: 시계열 단원에서 **시간으로 데이터를 나눈 것**과 같은 이유입니다.
시점 정확 조인은 그 규칙을 **분할점 하나가 아니라 모든 행에** 적용하는 거죠.

> **실험 3** · 누수가 성능에 하는 일
>
> **🎯 배울 것** — 데이터 누수는 **오프라인 점수를 올리고 실제 성능을 내립니다.** 그 둘을 같이 봐야 보입니다.
>
> **🔬 하는 일** — 이탈 데이터를 만들어 ① 순진한 조인 ② 시점 정확 조인으로 각각 규칙을 학습하고, **깨끗한 145일차 데이터**로 둘 다 평가합니다.

In [2]:
# 📗 원문의 데이터·조인·규칙을 옮깁니다
특징이름들 = ["days_since_last_purchase", "purchases_30d", "spend_30d"]
스냅샷일자 = list(range(7, 176, 7))      # 7일마다 배치 작업이 돕니다 (25회)
TTL일수 = 10


@dataclass(frozen=True)
class 이벤트:
    엔티티: str
    일: int
    금액: float


@dataclass(frozen=True)
class 특징정의:
    이름: str
    집계: str
    창일수: int
    기본값: float
    버전: int = 1

    def 계산(self, 사건들, 기준일):
        보이는것 = [e for e in 사건들 if e.일 <= 기준일]      # 기준일 이후는 절대 안 봅니다
        if self.창일수 is not None:
            보이는것 = [e for e in 보이는것 if e.일 > 기준일 - self.창일수]
        if self.집계 == "count":
            return float(len(보이는것))
        if self.집계 == "sum":
            return round(sum(e.금액 for e in 보이는것), 2)
        if self.집계 == "days_since_last":
            return float(기준일 - max(e.일 for e in 보이는것)) if 보이는것 else self.기본값
        raise ValueError(f"모르는 집계 {self.집계!r}")


정의들 = {
    "days_since_last_purchase": 특징정의("days_since_last_purchase", "days_since_last", None, 365.0),
    "purchases_30d": 특징정의("purchases_30d", "count", 30, 0.0),
    "spend_30d": 특징정의("spend_30d", "sum", 30, 0.0),
}


def 사건만들기(사용자수=300, 일수=180, 시드=11):
    # 📗 원문의 make_events 와 똑같이 둡니다 (숫자를 그대로 재현하려고요)
    rng = random.Random(시드)
    사건들 = []
    for u in range(사용자수):
        엔 = f"u{u:03d}"
        활동 = rng.uniform(0.08, 0.35)
        멈춤 = rng.randint(40, 170) if rng.random() < 0.5 else None     # 절반은 도중에 이탈
        for 일 in range(일수):
            if 멈춤 is not None and 일 >= 멈춤:
                break
            if rng.random() < 활동:
                사건들.append(이벤트(엔, 일, round(rng.uniform(5, 80), 2)))
    return 사건들


def 엔티티별(사건들):
    묶음 = defaultdict(list)
    for e in 사건들:
        묶음[e.엔티티].append(e)
    return 묶음


def 라벨만들기(사건들, 기준일들, 기간=30):
    묶음 = 엔티티별(사건들)
    라벨 = []
    for 엔 in sorted(묶음):
        for 일 in 기준일들:
            if not any(e.일 <= 일 for e in 묶음[엔]):
                continue
            미래구매 = any(일 < e.일 <= 일 + 기간 for e in 묶음[엔])
            라벨.append((엔, 일, 0 if 미래구매 else 1))     # 1 = 이탈
    return 라벨


def 백필(사건들, 일자들):
    행들 = []
    for 엔, 엔사건 in sorted(엔티티별(사건들).items()):
        for 일 in 일자들:
            값 = {n: 정의들[n].계산(엔사건, 일) for n in 특징이름들}
            행들.append((엔, 일, 값))
    return 행들


def 시점정확조인(라벨들, 행들, ttl=None):
    엔별 = defaultdict(list)
    for 엔, 일, 값 in 행들:
        엔별[엔].append((일, 값))
    일색인 = {}
    for 엔 in 엔별:
        엔별[엔].sort(key=lambda t: t[0])
        일색인[엔] = [t[0] for t in 엔별[엔]]
    결과 = []
    for 엔, 라벨일, y in 라벨들:
        일들 = 일색인.get(엔, [])
        위치 = bisect.bisect_right(일들, 라벨일) - 1          # 라벨일 이하 중 가장 최근
        맞음 = 엔별[엔][위치] if 위치 >= 0 else None
        if 맞음 and ttl is not None and 라벨일 - 맞음[0] > ttl:
            맞음 = None                                      # TTL 초과 -> 없는 셈
        결과.append((엔, 라벨일, y, 맞음[0] if 맞음 else None, 맞음[1] if 맞음 else None))
    return 결과


def 순진한조인(라벨들, 행들):
    최신 = {}
    for 엔, 일, 값 in 행들:
        if 엔 not in 최신 or 일 > 최신[엔][0]:
            최신[엔] = (일, 값)                               # 라벨 시점을 아예 안 봅니다
    return [(엔, 라벨일, y, *(최신.get(엔) or (None, None))) for 엔, 라벨일, y in 라벨들]


def 기본값채우기(값):
    if 값 is None:
        return {n: 정의들[n].기본값 for n in 특징이름들}
    return {n: 값.get(n, 정의들[n].기본값) for n in 특징이름들}


def 규칙학습(조인결과):
    X = [기본값채우기(r[4])["days_since_last_purchase"] for r in 조인결과]
    y = [r[2] for r in 조인결과]
    최적, 최고 = 0.0, -1.0
    for 임 in range(0, 120):                       # 📗 원문과 같은 후보 격자
        정확 = sum((x > 임) == (t == 1) for x, t in zip(X, y)) / len(y)
        if 정확 > 최고:
            최적, 최고 = float(임), 정확
    return 최적, 최고


def 규칙정확도(임, 조인결과):
    X = [기본값채우기(r[4])["days_since_last_purchase"] for r in 조인결과]
    y = [r[2] for r in 조인결과]
    return sum((x > 임) == (t == 1) for x, t in zip(X, y)) / len(y)


사건들 = 사건만들기()
오프라인행 = 백필(사건들, 스냅샷일자)
훈련라벨 = 라벨만들기(사건들, [100, 120, 140])
평가라벨 = 라벨만들기(사건들, [145])
평가조인 = 시점정확조인(평가라벨, 오프라인행, TTL일수)       # 평가는 늘 올바른 조인으로

print(f"사용자 {len(엔티티별(사건들))}명, 이벤트 {len(사건들)}건, 오프라인 행 {len(오프라인행)}개")
print(f"학습 라벨 {len(훈련라벨)}건 (100·120·140일차), 이탈률 {sum(l[2] for l in 훈련라벨)/len(훈련라벨):.2f}")
print(f"평가 라벨 {len(평가조인)}건 (145일차)\n")

순진 = 순진한조인(훈련라벨, 오프라인행)
정확 = 시점정확조인(훈련라벨, 오프라인행, TTL일수)
미래비율 = sum(1 for r in 순진 if r[3] and r[3] > r[1]) / len(순진)

print(f"{'학습에 쓴 조인':<20}{'학습된 규칙':>20}{'오프라인 정확도':>15}{'145일차 실제':>14}")
print("-" * 72)
for 라벨, 조인 in [("순진한 조인", 순진), ("시점 정확 조인", 정확)]:
    임, 오프 = 규칙학습(조인)
    print(f"{라벨:<19}{'idle > ' + format(임, '.0f') + '일':>21}{오프:>15.3f}{규칙정확도(임, 평가조인):>14.3f}")

print(f"\n순진한 조인이 라벨 날짜 이후 스냅샷을 쓴 비율: {미래비율:.0%}")
print(f"\n=> 윗줄이 오프라인에서 더 좋아 보입니다. 0.948 대 0.893.")
print(f"   그런데 실제 데이터에서는 더 나쁩니다. 0.763 대 0.876.")
print(f"\n   누수된 특징이 학습된 임곗값을 19일에서 53일로 밀어 올렸습니다.")
print(f"   53일은 미래를 본 상태에서나 말이 되는 숫자입니다.")
print(f"   실제 서빙에서는 아무도 53일을 기다려 주지 않거든요.")
print(f"\n   📝 이게 데이터 누수가 무서운 이유입니다.")
print(f"      더 나쁜 모델이 더 좋은 점수를 받습니다. 그래서 알아채기가 어렵습니다.")

사용자 299명, 이벤트 9224건, 오프라인 행 7475개
학습 라벨 897건 (100·120·140일차), 이탈률 0.35
평가 라벨 299건 (145일차)

학습에 쓴 조인                          학습된 규칙       오프라인 정확도      145일차 실제
------------------------------------------------------------------------
순진한 조인                        idle > 53일          0.948         0.763
시점 정확 조인                      idle > 19일          0.893         0.876

순진한 조인이 라벨 날짜 이후 스냅샷을 쓴 비율: 100%

=> 윗줄이 오프라인에서 더 좋아 보입니다. 0.948 대 0.893.
   그런데 실제 데이터에서는 더 나쁩니다. 0.763 대 0.876.

   누수된 특징이 학습된 임곗값을 19일에서 53일로 밀어 올렸습니다.
   53일은 미래를 본 상태에서나 말이 되는 숫자입니다.
   실제 서빙에서는 아무도 53일을 기다려 주지 않거든요.

   📝 이게 데이터 누수가 무서운 이유입니다.
      더 나쁜 모델이 더 좋은 점수를 받습니다. 그래서 알아채기가 어렵습니다.


> **👀 주목할 곳** — `오프라인 정확도` 열과 `145일차 실제` 열의 **순서가 뒤집힙니다.**
>
> **✅ 알게 된 사실** — 순진한 조인은 오프라인 0.948 로 이기고 실제 0.763 으로 집니다. 시점 정확 조인은 0.893 / 0.876 이고요.
누수가 임곗값을 19일 → 53일로 밀어 올렸습니다. 미래를 본 상태에서나 말이 되는 숫자죠.
**더 나쁜 모델이 더 좋은 점수를 받습니다.** 그래서 알아채기 어렵습니다.

> **🧩 이 절의 결론**
>
> 라벨에는 **그 시점에 존재했던 값만** 붙여야 합니다.
> 순진한 조인은 라벨 100%에 **미래 스냅샷**을 붙이고, 그 결과 오프라인 점수는 오르고 실제 성능은 떨어집니다.
>
> **➡️ 다음**
>
> 이건 괴리의 **한 종류**입니다. 📗 원문이 네 가지를 꼽습니다. 4절에서 정리합니다.

---

## 4. 학습-서빙 괴리의 네 가지 원인

<sub>📗 원문 "Causes of training-serving skew"</sub>

> 📗 구글 *Rules of Machine Learning* 은 학습-서빙 괴리를
> **학습 중 성능과 서빙 중 성능의 차이**로 정의합니다.
> 원인 셋을 꼽는데, 두 파이프라인의 **데이터 처리 차이**,
> 학습과 서빙 **사이의 데이터 변화**, 그리고 **피드백 루프**입니다.

특징 파이프라인에서는 **구체적인 원인 넷**이 반복해서 나타납니다.

| 원인 | 무슨 일이 | 이 노트북의 예 |
| --- | --- | --- |
| **코드가 두 벌** | 학습과 서빙이 **다른 코드**로 같은 특징을 계산 | 서빙 창에 `now - 30` 일이 **하나 더** 들어감 |
| **묵은 특징** | 서빙이 읽는 값의 **나이**가 학습 때와 다름 | 학습은 주간 스냅샷, 새 서비스는 **그 자리에서 계산** |
| **다른 기본값** | 결측값을 **서로 다르게** 채움 | 학습은 365, 서빙은 **−1** |
| **시간 여행** | 학습이 **라벨 이후** 값을 읽음 | **3절의 순진한 조인** |

### 🔑 해법은 📗 원문의 32번 규칙입니다

> **학습 파이프라인과 서빙 파이프라인 사이에 코드를 재사용하라.**

피처 스토어는 그 규칙을 **정의 하나를 양쪽이 호출하는 것**으로 구현합니다.
그게 5절의 레지스트리입니다.

### 🔑 그리고 넷은 서로 다른 흔적을 남깁니다

| 원인 | 어디서 잡히나 |
| --- | --- |
| 시간 여행 | 오프라인은 좋은데 실제가 나쁨 (3절) |
| 코드가 두 벌 · 다른 기본값 | **"두 번째 코드 경로 vs 즉석 계산"** 비교 (8절) |
| 묵은 특징 | **"즉석 계산 vs 학습 조인"** 비교 (8절) |

**어느 비교가 깨졌느냐가 범인을 가리킵니다.** 그래서 8절에서 세 쌍을 다 비교합니다.

> **🧩 이 절의 결론**
>
> 괴리의 원인은 넷입니다. **코드 두 벌 · 묵은 특징 · 다른 기본값 · 시간 여행.**
> 넷 다 '같은 특징을 두 가지 방법으로 계산한' 결과고, 해법은 **정의를 한 곳에 두는 것**입니다.
>
> **➡️ 다음**
>
> 그 '한 곳' 을 코드로 만들어 봅니다. 5절의 레지스트리입니다.

---

## 5. 특징 레지스트리 — 정의는 하나

<sub>📗 원문 "A feature registry"</sub>

레지스트리는 **특징 정의의 목록**입니다.
항목마다 이름 · 엔티티 · 원본 · 변환 · 창 · 기본값 · TTL · 소유자 · 버전 · 설명을 담습니다.

### 📗 안전하게 쓰는 두 규칙

1. **양쪽 경로가 레지스트리에서 정의를 읽습니다.**
   어떤 서비스도 변환 로직의 **사본을 따로 갖지 않습니다**
2. **정의를 바꾸면 새 버전이 생깁니다.**
   옛 버전은 남아 있어서 **옛 모델이 학습한 특징을 계속 쓸 수 있습니다**

### 🔑 버전 고정이 왜 필요한가

모델은 `spend_30d@1` 같은 **참조로 정의를 고정**합니다.

> 📗 `@` 없이 이름만 쓰면 **최신 버전**을 읽습니다.
> `@` 는 있는데 버전 번호가 없으면 **`ValueError` 를 냅니다.**
> 오타 하나가 조용히 최신 버전을 읽어 버리는 일을 막으려고요.

그리고 이 참조는 **저장되는 행의 키**이기도 합니다.
백필도, 온라인 스토어도, 서빙 읽기도 **전부 고정된 정의**를 씁니다.

> **실험 5** · 레지스트리 — 정의 하나, 독자 둘
>
> **🎯 배울 것** — '코드 재사용' 이 **어떻게 코드가 되는지** 봅니다.
>
> **🔬 하는 일** — 레지스트리를 만들고, ① 학습 경로 ② 서빙 경로가 **같은 정의 객체**를 호출하는지 확인합니다. 버전 고정도 시험합니다.

In [3]:
class 레지스트리:
    def __init__(self):
        self._정의 = {}

    def 등록(self, 정의):
        키 = (정의.이름, 정의.버전)
        기존 = self._정의.get(키)
        if 기존 is not None and 기존 != 정의:
            raise ValueError(f"{정의.이름}@{정의.버전} 는 이미 다른 정의로 등록돼 있습니다")
        self._정의[키] = 정의
        return self

    def 찾기(self, 참조):
        if "@" not in 참조:
            버전들 = [v for (n, v) in self._정의 if n == 참조]
            if not 버전들:
                raise KeyError(f"{참조} 가 레지스트리에 없습니다")
            return self._정의[(참조, max(버전들))]          # 버전 생략 -> 최신
        이름, _, 버전글 = 참조.partition("@")
        if not 버전글.isdigit():
            raise ValueError(f"{참조!r}: @ 뒤에 버전 번호가 필요합니다 (오타가 최신을 읽는 걸 막습니다)")
        return self._정의[(이름, int(버전글))]

    def 계산(self, 참조들, 사건들, 기준일):
        return {r: self.찾기(r).계산(사건들, 기준일) for r in 참조들}

    def 기본값채우기(self, 참조들, 값):
        if 값 is None:
            return {r: self.찾기(r).기본값 for r in 참조들}
        return {r: 값.get(r, self.찾기(r).기본값) for r in 참조들}


등록소 = 레지스트리()
for 정의 in 정의들.values():
    등록소.등록(정의)

print("등록된 정의")
print(f"{'이름':<28}{'버전':>5}{'집계':>18}{'창':>7}{'기본값':>9}")
print("-" * 70)
for (이름, 버전), 정의 in sorted(등록소._정의.items()):
    창 = f"{정의.창일수}일" if 정의.창일수 else "전체"
    print(f"{이름:<27}{버전:>5}{정의.집계:>18}{창:>7}{정의.기본값:>9.0f}")

print("\n\n=== 같은 정의를 양쪽이 호출하는지 확인 ===")
한명 = 엔티티별(사건들)["u001"]
학습쪽 = 등록소.계산(특징이름들, 한명, 140)     # 학습: 과거 시점을 물어봄
서빙쪽 = 등록소.계산(특징이름들, 한명, 140)     # 서빙: 같은 정의, 같은 시점
print(f"   학습 경로가 140일에 계산: {학습쪽}")
print(f"   서빙 경로가 140일에 계산: {서빙쪽}")
print(f"   두 결과가 같은가: {학습쪽 == 서빙쪽}")
print(f"   같은 정의 객체를 쓰는가: {등록소.찾기('spend_30d') is 등록소.찾기('spend_30d@1')}")

print("\n\n=== 레지스트리가 막아 주는 실수들 ===")
for 설명, 하는일 in [
    ("같은 이름·같은 버전으로 다른 정의 등록",
     lambda: 등록소.등록(특징정의("spend_30d", "sum", 7, 0.0, 1))),
    ("버전 번호 없이 `@` 만 (오타)", lambda: 등록소.찾기("spend_30d@")),
    ("없는 특징 참조", lambda: 등록소.찾기("spend_90d")),
]:
    try:
        하는일()
        print(f"   ✘ {설명}: 통과해 버렸습니다")
    except (ValueError, KeyError) as 오류:
        print(f"   ○ {설명}")
        print(f"       -> {type(오류).__name__}: {오류}")

print(f"\n=> 세 번째 줄이 특히 중요합니다. `spend_30d@` 같은 오타가")
print(f"   조용히 최신 버전을 읽어 버리면 옛 모델이 모르는 사이에 다른 특징을 받습니다.")
print(f"   그래서 일부러 터뜨립니다.")

등록된 정의
이름                             버전                집계      창      기본값
----------------------------------------------------------------------
days_since_last_purchase       1   days_since_last     전체      365
purchases_30d                  1             count    30일        0
spend_30d                      1               sum    30일        0


=== 같은 정의를 양쪽이 호출하는지 확인 ===
   학습 경로가 140일에 계산: {'days_since_last_purchase': 22.0, 'purchases_30d': 3.0, 'spend_30d': 102.97}
   서빙 경로가 140일에 계산: {'days_since_last_purchase': 22.0, 'purchases_30d': 3.0, 'spend_30d': 102.97}
   두 결과가 같은가: True
   같은 정의 객체를 쓰는가: True


=== 레지스트리가 막아 주는 실수들 ===
   ○ 같은 이름·같은 버전으로 다른 정의 등록
       -> ValueError: spend_30d@1 는 이미 다른 정의로 등록돼 있습니다
   ○ 버전 번호 없이 `@` 만 (오타)
       -> ValueError: 'spend_30d@': @ 뒤에 버전 번호가 필요합니다 (오타가 최신을 읽는 걸 막습니다)
   ○ 없는 특징 참조
       -> KeyError: 'spend_90d 가 레지스트리에 없습니다'

=> 세 번째 줄이 특히 중요합니다. `spend_30d@` 같은 오타가
   조용히 최신 버전을 읽어 버리면 옛 모델이 모르는 사이에 다른 특징을 받습니다.
   그래서 일부러 터뜨립니다.


> **👀 주목할 곳** — `두 결과가 같은가: True` 와, 마지막 세 줄의 `->` 오류 메시지들.
>
> **✅ 알게 된 사실** — 양쪽 경로가 **같은 정의 객체**를 호출하니 결과가 같을 수밖에 없습니다. 4절의 '코드 두 벌' 이 구조적으로 불가능해집니다.
그리고 레지스트리는 **조용한 실수를 터뜨립니다.** `spend_30d@` 오타가 최신 버전을 읽는 것 같은 일이요.

> **🧩 이 절의 결론**
>
> 레지스트리는 **정의를 한 곳에 두고 양쪽이 읽게** 합니다. 32번 규칙의 구현이죠.
> 그리고 **버전을 고정**해서 옛 모델이 학습한 특징을 계속 받게 합니다.
>
> **➡️ 다음**
>
> 정의가 생겼으니 **과거 값을 채워야** 합니다. 새 특징은 이력이 없으니까요. 6절입니다.

---

## 6. 백필 — 과거를 채우기

<sub>📗 원문 "Backfills"</sub>

새 특징에는 **이력이 없습니다.** 그런데 학습에는 **모든 옛 라벨 시점의 값**이 필요하죠.

**백필**은 정의를 **과거 날짜마다 실행**해서 엔티티·날짜별로 한 행씩 씁니다.
각 실행은 **그 날짜 이하의 이벤트만** 읽고요.

### 📗 비용과 누수, 두 가지를 조심합니다

**비용**은 **엔티티 수 × 날짜 수**로 커집니다.
이 노트북은 사용자 299명 × 스냅샷 25일 = **7475 행**입니다.
10절에서 날짜 간격을 줄이면 비용이 어떻게 뛰는지 봅니다.

**누수**는 2절에서 예고한 것입니다.

> 📗 오늘 돌린 백필은 옛 이벤트 시각으로 행을 쓰지만,
> **늦게 도착했거나 나중에 수정된 데이터**를 읽을 수 있습니다.
> 엄격한 시점 정확 조인은 **생성 시각도** 라벨 시점 이전인지 확인합니다.

> **실험 6** · 백필과 '그 날짜 이후는 안 본다' 확인
>
> **🎯 배울 것** — 백필이 정말 **미래를 안 보는지** 확인합니다. 이게 안 지켜지면 3절의 누수가 다시 생깁니다.
>
> **🔬 하는 일** — 같은 사용자의 같은 특징을 날짜별로 백필하고, 그 값들이 **그 날짜까지의 이벤트만으로** 설명되는지 대조합니다.

In [4]:
오프라인행 = 백필(사건들, 스냅샷일자)
print(f"백필: 사용자 {len(엔티티별(사건들))}명 × 스냅샷 {len(스냅샷일자)}일 = {len(오프라인행)} 행\n")

한명 = 엔티티별(사건들)["u001"]
구매일 = sorted(e.일 for e in 한명)
print(f"u001 의 구매 일자: {구매일[:12]} ... (총 {len(구매일)}회)\n")
print(f"{'스냅샷 날짜':>11}{'days_since':>12}{'purchases_30d':>15}{'spend_30d':>12}   직접 세어 확인")
print("-" * 82)
for 일 in (28, 56, 84, 112, 140):
    값 = dict((엔, 일2, v) for 엔, 일2, v in 오프라인행 if 엔 == "u001" and 일2 == 일).popitem()[1] \
        if False else [v for 엔, 일2, v in 오프라인행 if 엔 == "u001" and 일2 == 일][0]
    # 백필 결과와 별개로, '그 날짜까지의 이벤트' 만으로 직접 다시 셉니다
    과거만 = [e for e in 한명 if e.일 <= 일]
    창안 = [e for e in 과거만 if e.일 > 일 - 30]
    확인 = (len(창안) == 값["purchases_30d"]
          and abs(round(sum(e.금액 for e in 창안), 2) - 값["spend_30d"]) < 1e-6)
    print(f"{일:>11}{값['days_since_last_purchase']:>12.0f}{값['purchases_30d']:>15.0f}"
          f"{값['spend_30d']:>12.2f}   {'일치' if 확인 else '불일치!'}")

print(f"\n=> `purchases_30d` 열을 보세요. 날짜가 가도 단조롭게 늘지 않습니다.")
print(f"   30일 창이라 옛 구매는 창 밖으로 빠져나가거든요.")
print(f"   이게 백필이 '그 날짜 기준' 으로 계산했다는 증거입니다.")
print(f"   최신 값만 복사했다면 모든 날짜에 같은 숫자가 찍혔을 겁니다.")

print(f"\n\n=== 백필 비용은 엔티티 수 × 날짜 수 ===")
print(f"{'스냅샷 간격':>12}{'날짜 수':>9}{'백필 행 수':>12}{'7일 대비':>11}")
print("-" * 46)
기준행수 = None
for 간격 in (7, 3, 1):
    일자 = list(range(간격, 176, 간격))
    행수 = (len(엔티티별(사건들))) * len(일자)
    기준행수 = 기준행수 or 행수
    print(f"{간격:>11}일{len(일자):>9}{행수:>12,}{행수 / 기준행수:>10.1f}배")
print(f"\n   10절에서 이 비용을 신선도와 맞바꿔 봅니다.")

백필: 사용자 299명 × 스냅샷 25일 = 7475 행

u001 의 구매 일자: [0, 3, 6, 17, 19, 24, 25, 28, 32, 34, 37, 43] ... (총 35회)

     스냅샷 날짜  days_since  purchases_30d   spend_30d   직접 세어 확인
----------------------------------------------------------------------------------
         28           0              8      286.70   일치
         56           6              8      335.83   일치
         84          12              9      442.77   일치
        112           1              9      364.78   일치
        140          22              3      102.97   일치

=> `purchases_30d` 열을 보세요. 날짜가 가도 단조롭게 늘지 않습니다.
   30일 창이라 옛 구매는 창 밖으로 빠져나가거든요.
   이게 백필이 '그 날짜 기준' 으로 계산했다는 증거입니다.
   최신 값만 복사했다면 모든 날짜에 같은 숫자가 찍혔을 겁니다.


=== 백필 비용은 엔티티 수 × 날짜 수 ===
      스냅샷 간격     날짜 수      백필 행 수      7일 대비
----------------------------------------------
          7일       25       7,475       1.0배
          3일       58      17,342       2.3배
          1일      175      52,325       7.0배

   10절에서 이 비용을 신선도와 맞바꿔 봅니다.


> **👀 주목할 곳** — `purchases_30d` 열이 날짜 따라 **오르내립니다.**
>
> **✅ 알게 된 사실** — 30일 창이라 옛 구매가 창 밖으로 빠져나가기 때문입니다.
이게 백필이 **각 날짜 기준으로 다시 계산**했다는 증거입니다. 최신 값을 복사했다면 전 날짜가 같은 숫자였겠죠.
그리고 비용은 **엔티티 × 날짜**입니다. 간격을 7일에서 1일로 줄이면 7배가 됩니다.

> **🧩 이 절의 결론**
>
> 백필은 정의를 **과거 날짜마다 다시 실행**합니다. 각 실행은 그 날짜 이하만 읽고요.
> 비용은 **엔티티 수 × 날짜 수**로 커집니다.
>
> **➡️ 다음**
>
> 오프라인 스토어가 찼습니다. 이제 **서빙 쪽**을 채울 차례입니다. 7절입니다.

---

## 7. 온라인 스토어와 TTL

<sub>📗 원문 "The online store and materialization"</sub>

온라인 스토어는 **엔티티당 행 하나**만 둡니다. 그리고 **TTL 보다 오래된 행은 거부**합니다.

**머티리얼라이제이션**은 오프라인 스토어에서 최신 값을 복사해 오는 작업인데,
**작업 시각 이하의 행만** 씁니다. (아니면 또 시간 여행이죠.)

### 🔑 TTL 이 하는 일

> 📗 TTL 10일로 두면,
> **145일 읽기**는 140일 행을 돌려줍니다 (5일 묵음, 통과).
> **160일 읽기**는 — 새 머티리얼라이제이션이 없었다면 —
> 그 행이 **20일** 묵었으므로 **기본값**을 돌려줍니다.

마지막 줄이 중요합니다. **낡은 값을 조용히 주는 것보다 기본값을 주는 게 낫습니다.**
기본값은 **양쪽 경로가 레지스트리에서 똑같이** 가져오니까요(4절의 '다른 기본값' 방지).

> **실험 7** · 온라인 스토어와 TTL 만료
>
> **🎯 배울 것** — TTL 이 **낡은 값 대신 기본값**을 주는 순간을 직접 봅니다.
>
> **🔬 하는 일** — 온라인 스토어를 만들어 140일까지 머티리얼라이즈하고, 145일과 160일에 같은 사용자를 읽어 봅니다.

In [5]:
class 온라인스토어:
    def __init__(self, ttl일수):
        self.ttl일수 = ttl일수
        self._최신 = {}

    def 쓰기(self, 엔, 일, 값):
        현재 = self._최신.get(엔)
        if 현재 is None or 일 >= 현재[0]:
            self._최신[엔] = (일, 값)

    def 읽기(self, 엔, 지금):
        행 = self._최신.get(엔)
        if 행 is None or 행[0] > 지금 or 지금 - 행[0] > self.ttl일수:
            return None                                   # 없거나, 미래거나, 너무 묵음
        return dict(행[1])


def 머티리얼라이즈(행들, 스토어, 작업일):
    쓴수 = 0
    for 엔, 일, 값 in 행들:
        if 일 <= 작업일:                                   # 작업 시각 이후 행은 안 씁니다
            스토어.쓰기(엔, 일, 값)
            쓴수 += 1
    return 쓴수


스토어 = 온라인스토어(TTL일수)
쓴수 = 머티리얼라이즈(오프라인행, 스토어, 140)
print(f"140일 기준으로 머티리얼라이즈: {쓴수}행 처리, 엔티티 {len(스토어._최신)}개 저장")
print(f"TTL = {TTL일수}일\n")

print(f"{'읽는 시점':>10}{'저장된 행 날짜':>14}{'행 나이':>9}{'TTL 통과':>10}   돌려주는 값")
print("-" * 96)
for 지금 in (142, 145, 150, 155, 160):
    행일 = 스토어._최신["u001"][0]
    나이 = 지금 - 행일
    값 = 스토어.읽기("u001", 지금)
    통과 = 값 is not None
    보여줄 = 등록소.기본값채우기(특징이름들, 값)
    꼬리 = "" if 통과 else "  <- 만료, 기본값"
    print(f"{지금:>10}{행일:>14}{나이:>8}일{'예' if 통과 else '아니오':>10}   "
          f"{ {k: round(v, 1) for k, v in 보여줄.items()} }{꼬리}")

print(f"\n=> 150일과 155일 사이에서 끊깁니다. 행이 140일 것이니 TTL 10일을 넘기는 거죠.")
print(f"   만료되면 `days_since_last_purchase` 가 365 로 돌아옵니다.")
print(f"   레지스트리에 적힌 기본값이에요. 학습 쪽도 같은 365 를 씁니다.")
print(f"   그래서 4절의 '다른 기본값' 괴리가 안 생깁니다.")
print(f"\n   📝 TTL 이 없으면 160일에 20일 묵은 값을 조용히 돌려줍니다.")
print(f"      그게 더 위험합니다. 틀린 걸 모르니까요.")

140일 기준으로 머티리얼라이즈: 5980행 처리, 엔티티 299개 저장
TTL = 10일

     읽는 시점      저장된 행 날짜     행 나이    TTL 통과   돌려주는 값
------------------------------------------------------------------------------------------------
       142           140       2일         예   {'days_since_last_purchase': 22.0, 'purchases_30d': 3.0, 'spend_30d': 103.0}
       145           140       5일         예   {'days_since_last_purchase': 22.0, 'purchases_30d': 3.0, 'spend_30d': 103.0}
       150           140      10일         예   {'days_since_last_purchase': 22.0, 'purchases_30d': 3.0, 'spend_30d': 103.0}
       155           140      15일       아니오   {'days_since_last_purchase': 365.0, 'purchases_30d': 0.0, 'spend_30d': 0.0}  <- 만료, 기본값
       160           140      20일       아니오   {'days_since_last_purchase': 365.0, 'purchases_30d': 0.0, 'spend_30d': 0.0}  <- 만료, 기본값

=> 150일과 155일 사이에서 끊깁니다. 행이 140일 것이니 TTL 10일을 넘기는 거죠.
   만료되면 `days_since_last_purchase` 가 365 로 돌아옵니다.
   레지스트리에 적힌 기본값이에요. 학습 쪽도 같은 365 를 씁니다.
   그래서 4절의 '다른 

> **👀 주목할 곳** — `TTL 통과` 열이 '예' 에서 '아니오' 로 바뀌는 지점, 그리고 그때 값이 **365** 가 되는 것.
>
> **✅ 알게 된 사실** — TTL 은 **낡은 값을 조용히 주는 대신 기본값을 주게** 합니다.
그 기본값은 레지스트리에서 오고, **학습 쪽도 같은 값을 씁니다.** 4절의 '다른 기본값' 괴리가 구조적으로 막힙니다.

> **🧩 이 절의 결론**
>
> 온라인 스토어는 **엔티티당 최신 값 하나**를 두고 **TTL 로 나이를 제한**합니다.
> 만료되면 레지스트리의 기본값을 주는데, **학습 쪽도 같은 기본값**을 쓰니 괴리가 안 생깁니다.
>
> **➡️ 다음**
>
> 부품이 다 모였습니다. 이제 **진짜 같은 값이 나오는지** 확인해야죠.
> 그리고 다르다면 **어디서** 달라졌는지도요. 8절입니다.

---

## 8. 괴리 탐지 — 세 쌍을 비교하면 범인이 좁혀집니다

<sub>📗 원문 "Detecting skew"</sub>

> 📗 29번 규칙: **서빙 시점의 특징을 기록해서 학습에 쓰라.**
> 작은 표본만 기록해도 일관성을 확인할 수 있습니다.

기록된 값을 **같은 엔티티·같은 시점에 대해 학습 경로가 내놓는 값**과 비교합니다.
특징마다 **불일치 비율**과 **평균 절대 차이**를 보고하고요.

### 🔑 원인을 찾으려면 세 쌍을 비교합니다

| # | 비교 | 무엇을 가려내나 | 기대 |
| --- | --- | --- | --- |
| 1 | **온라인 스토어 읽기** vs 학습 조인 | 시스템 전체 | **괴리 0** 이어야 함 |
| 2 | **즉석 계산** vs 학습 조인 | **묵음**만 떼어 봄 | 차이 날 수 있음 |
| 3 | **두 번째 코드 경로** vs 즉석 계산 | **코드 차이**만 떼어 봄 (창·기본값) | 차이 나면 버그 |

### 🔑 왜 이 순서인가

각 비교가 **변수를 하나씩만** 다르게 둡니다.

- 1번은 정의도 같고 스냅샷도 같습니다 → 다르면 **시스템이 깨진 것**
- 2번은 정의는 같은데 **시점이 다릅니다** → 차이는 **전부 묵음 탓**
- 3번은 시점은 같은데 **코드가 다릅니다** → 차이는 **전부 코드 탓**

통제 실험이죠. 그래서 **어느 쌍이 깨졌느냐가 범인을 가리킵니다.**

> 📗 `detect_skew` 는 두 쪽의 **키가 다르거나 비어 있으면 `ValueError`** 를 냅니다.
> 요청이 빠지면 괴리를 숨길 수 있고, 요청이 없는 검사는 아무것도 못 보여 주니까요.

> **실험 8** · 세 쌍 비교로 범인 좁히기
>
> **🎯 배울 것** — 괴리 수치 하나로는 원인을 모릅니다. **변수를 하나씩만 바꿔** 비교하면 알 수 있습니다.
>
> **🔬 하는 일** — 141~150일 요청을 기록하고, 세 가지 특징 계산 방식을 쌍으로 비교합니다. 일부러 버그가 있는 '옛 서비스' 코드도 넣습니다.

In [6]:
def 괴리검사(학습쪽, 서빙쪽, 특징들, 허용=1e-6, 최대불일치=0.01):
    if set(학습쪽) != set(서빙쪽):
        틀린수 = len(set(학습쪽) ^ set(서빙쪽))
        raise ValueError(f"{틀린수}개 키가 한쪽에만 있습니다. 같은 요청을 기록해 비교하세요")
    if not 학습쪽:
        raise ValueError("비교할 요청이 없습니다")
    키들 = sorted(학습쪽)
    보고 = []
    for f in 특징들:
        차이 = [abs(학습쪽[k][f] - 서빙쪽[k][f]) for k in 키들]
        불일치 = sum(1 for d in 차이 if d > 허용)
        비율 = 불일치 / len(키들)
        보고.append((f, len(키들), 비율, sum(차이) / len(차이), 비율 > 최대불일치))
    return 보고


def 옛서비스특징(사건들, 지금):
    # ⚠️ 일부러 버그를 둡니다 — 📗 원문의 legacy_serving_features
    최근 = [e for e in 사건들 if 지금 - 30 <= e.일 <= 지금]      # 창에 하루가 더 들어감
    과거 = [e.일 for e in 사건들 if e.일 <= 지금]
    return {
        "days_since_last_purchase": float(지금 - max(과거)) if 과거 else -1.0,   # 기본값이 -1
        "purchases_30d": float(len(최근)),
        "spend_30d": round(sum(e.금액 for e in 최근), 2),
    }


묶음 = 엔티티별(사건들)
사용자목록 = sorted(묶음) + [f"신규{i:02d}" for i in range(20)]     # 이력 없는 신규 20명
전체요청 = [(u, d) for u in 사용자목록 for d in range(141, 151)]
random.Random(3).shuffle(전체요청)                                # 신규 사용자도 섞이게
요청들 = sorted(전체요청[:383])
신규수 = sum(1 for u, _ in 요청들 if u.startswith("신규"))

# ① 온라인 스토어 읽기 (정의 하나, 같은 스냅샷)
온라인값 = {}
for 엔, 일 in 요청들:
    s = 온라인스토어(TTL일수)
    머티리얼라이즈([r for r in 오프라인행 if r[0] == 엔], s, 일)
    온라인값[(엔, 일)] = 등록소.기본값채우기(특징이름들, s.읽기(엔, 일))

# ② 학습 조인이 내놓는 값 (같은 스냅샷을 시점 정확 조인으로)
학습값 = {}
조인 = 시점정확조인([(엔, 일, 0) for 엔, 일 in 요청들], 오프라인행, TTL일수)
for 엔, 일, _, _, 값 in 조인:
    학습값[(엔, 일)] = 등록소.기본값채우기(특징이름들, 값)

# ③ 즉석 계산 (같은 정의, 다른 시점 = 묵음만 다름)
즉석값 = {(엔, 일): 등록소.기본값채우기(특징이름들, 등록소.계산(특징이름들, 묶음.get(엔, []), 일))
        for 엔, 일 in 요청들}

# ④ 옛 서비스 (같은 시점, 다른 코드 = 코드 차이만 다름)
옛값 = {(엔, 일): 옛서비스특징(묶음.get(엔, []), 일) for 엔, 일 in 요청들}

print(f"=== 기록된 요청 {len(요청들)}건, 141~150일 (그중 {신규수}건이 이력 없는 신규 사용자) ===\n")
for 제목, 왼쪽, 오른쪽, 떼어낸것 in [
    ("① 온라인 스토어 읽기  vs  학습 조인", 온라인값, 학습값, "정의도 같고 스냅샷도 같음"),
    ("② 즉석 계산          vs  학습 조인", 즉석값, 학습값, "묵음만 다름"),
    ("③ 옛 서비스 코드      vs  즉석 계산", 옛값, 즉석값, "코드 차이만 다름"),
]:
    print(f"  {제목}   ({떼어낸것})")
    for f, n, 비율, 평균차, 깃발 in 괴리검사(왼쪽, 오른쪽, 특징이름들):
        print(f"    {'괴리' if 깃발 else '정상'} {f:<28}불일치 {비율:>6.1%}   평균 |차이| {평균차:>8.2f}")
    print()

print("=> ① 이 전부 '정상' 입니다. 같은 정의로 같은 스냅샷을 읽으니 당연합니다.")
print("   이게 깨지면 시스템 자체가 고장난 것이니 제일 먼저 봐야 합니다.")
print("\n   ② 는 불일치가 큽니다. 하지만 버그가 아닙니다. 시점이 다를 뿐이에요.")
print("      학습은 주간 스냅샷(최대 6일 묵은 값)을 썼고 즉석 계산은 오늘 값입니다.")
print("      9절에서 이게 성능에 해로운지 따로 재 봅니다.")
print("\n   ③ 은 진짜 버그입니다. 창이 하루 더 들어가서 개수가 어긋나고,")
print("      신규 사용자 기본값이 -1 이라 365 와 366 만큼 벌어집니다.")
print("      그래서 `days_since_last_purchase` 는 불일치가 6.8% 뿐인데 평균 차이가 24.85 나 됩니다.")

=== 기록된 요청 383건, 141~150일 (그중 26건이 이력 없는 신규 사용자) ===

  ① 온라인 스토어 읽기  vs  학습 조인   (정의도 같고 스냅샷도 같음)
    정상 days_since_last_purchase    불일치   0.0%   평균 |차이|     0.00
    정상 purchases_30d               불일치   0.0%   평균 |차이|     0.00
    정상 spend_30d                   불일치   0.0%   평균 |차이|     0.00

  ② 즉석 계산          vs  학습 조인   (묵음만 다름)
    괴리 days_since_last_purchase    불일치  78.6%   평균 |차이|     2.57
    괴리 purchases_30d               불일치  30.8%   평균 |차이|     0.40
    괴리 spend_30d                   불일치  38.6%   평균 |차이|    17.21

  ③ 옛 서비스 코드      vs  즉석 계산   (코드 차이만 다름)
    괴리 days_since_last_purchase    불일치   6.8%   평균 |차이|    24.85
    괴리 purchases_30d               불일치  17.2%   평균 |차이|     0.17
    괴리 spend_30d                   불일치  17.2%   평균 |차이|     7.09

=> ① 이 전부 '정상' 입니다. 같은 정의로 같은 스냅샷을 읽으니 당연합니다.
   이게 깨지면 시스템 자체가 고장난 것이니 제일 먼저 봐야 합니다.

   ② 는 불일치가 큽니다. 하지만 버그가 아닙니다. 시점이 다를 뿐이에요.
      학습은 주간 스냅샷(최대 6일 묵은 값)을 썼고 즉석 계산은 오늘 값입니다.
      9절에서 이게 성능에 해로운지 따로 재 봅니다.

   ③ 은 진짜 버그입니다. 

> **👀 주목할 곳** — ③ 의 `days_since_last_purchase` 줄. **불일치 비율은 낮은데 평균 차이가 큽니다.**
>
> **✅ 알게 된 사실** — 신규 사용자에게만 틀리는데, 틀릴 때 **365 대 −1** 로 크게 틀리기 때문입니다.
**비율과 크기를 둘 다 봐야** 하는 이유입니다. 비율만 보면 '5%뿐이네' 하고 넘어갑니다.
그리고 ① 이 정상이라는 게 **시스템이 멀쩡하다는 증거**입니다. 여기가 깨지면 제일 먼저 고쳐야 합니다.

> **🧩 이 절의 결론**
>
> 괴리는 **세 쌍을 비교**해서 원인을 좁힙니다. 각 비교가 변수를 하나씩만 다르게 두니까요.
> ①이 깨지면 시스템 고장, ②는 묵음, ③은 코드 차이입니다.
> 그리고 **불일치 비율과 평균 차이를 둘 다** 봐야 합니다.
>
> **➡️ 다음**
>
> 그런데 ② 에서 불일치가 78.6% 나왔습니다. 큰 숫자죠.
> ⚠️ **그게 성능을 망치고 있을까요?** 9절에서 재 봅니다. 결과가 예상과 다릅니다.

---

## 9. ⚠️ 괴리가 크다고 성능이 나쁜 건 아닙니다

<sub>📗 원문 "Fresher data is not automatically better here" + ✨ 측정</sub>

8절의 ② 번 비교에서 **불일치 78.6%** 가 나왔습니다.
서빙이 즉석 계산을 쓰면 학습 때 값과 **열에 여덟은 다르다**는 뜻이죠.

📗 원문은 이렇게 설명합니다.

> 더 신선한 값이 **여기서 자동으로 더 나은 건 아닙니다.**
> 모델이 **최대 6일 묵은 값**으로 배웠기 때문입니다.

**말이 됩니다.** 모델은 "6일쯤 묵은 값" 의 분포를 보고 임곗값을 맞췄으니까요.
그런데 **정말 그런지 재 봐야** 압니다.

> **실험 9** · ⚠️ 신선한 값이 정확도를 떨어뜨리나
>
> **🎯 배울 것** — 괴리 수치가 **성능 손실을 뜻하는지** 직접 확인합니다.
>
> **🔬 하는 일** — 주간 스냅샷으로 학습한 규칙을 가져와, 서빙 때 ⓐ 스냅샷 값 ⓑ 즉석 계산 값을 쓰는 두 경우의 정확도를 여러 날짜에서 비교합니다.

In [7]:
# ✨ 추가 — 78.6% 불일치가 정확도를 떨어뜨리나
학습조인 = 시점정확조인(훈련라벨, 오프라인행, TTL일수)
임곗값, _ = 규칙학습(학습조인)
print(f"주간 스냅샷으로 학습한 규칙: days_since_last_purchase > {임곗값:.0f}일")
print(f"(모델은 최대 6일 묵은 값들을 보고 이 임곗값을 맞췄습니다)\n")
print(f"{'평가일':>8}{'(a) 스냅샷 값':>16}{'(b) 즉석 계산':>16}{'차이':>10}   더 나은 쪽")
print("-" * 68)
A = B = 0.0
n = 0
for 일 in (142, 145, 148, 151, 154, 157):
    라 = 라벨만들기(사건들, [일])
    j = 시점정확조인(라, 오프라인행, TTL일수)
    a = 규칙정확도(임곗값, j)
    # 같은 라벨에 즉석 계산 값을 붙입니다
    j신선 = [(엔, d, y, d, 등록소.계산(특징이름들, 묶음.get(엔, []), d))
           for 엔, d, y, _, _ in j]
    b = 규칙정확도(임곗값, j신선)
    A += a
    B += b
    n += 1
    더나은 = "즉석" if b > a else ("스냅샷" if a > b else "같음")
    print(f"{일:>8}{a:>16.3f}{b:>16.3f}{b - a:>+10.3f}   {더나은}")
print("-" * 68)
print(f"{'평균':>8}{A/n:>16.3f}{B/n:>16.3f}{(B-A)/n:>+10.3f}")

print(f"\n=> ⚠️ 즉석 계산이 한 번도 지지 않았습니다. 평균 {(B-A)/n:+.3f}.")
print(f"\n   📗 원문은 '더 신선한 게 자동으로 더 낫진 않다' 고 조심스럽게 말합니다.")
print(f"   측정해 보니 여기서는 신선한 쪽이 조금 낫거나 같았습니다.")
print(f"\n   📝 그래서 이 절의 교훈은 '신선한 게 낫다' 가 아닙니다. 이겁니다.")
print(f"\n      괴리 수치는 '두 쪽이 다르다' 만 말해 줍니다.")
print(f"      '어느 쪽이 맞다' 는 안 말해 줍니다.")
print(f"\n   불일치 78.6% 를 보고 '서빙이 틀렸다' 고 단정하면 안 됩니다.")
print(f"   여기서는 오히려 서빙 쪽이 나았으니까요.")
print(f"   괴리를 발견하면 어느 쪽이 맞는지는 정확도로 따로 재야 합니다.")
print(f"\n   💡 다만 8절 ③ 번(코드 차이)은 다릅니다. 그건 명백한 버그예요.")
print(f"      창에 하루가 더 들어간 것과 기본값 -1 은 의도된 적이 없는 차이니까요.")
print(f"      ② 번 묵음은 설계 선택이고, ③ 번 코드 차이는 버그입니다.")

주간 스냅샷으로 학습한 규칙: days_since_last_purchase > 19일
(모델은 최대 6일 묵은 값들을 보고 이 임곗값을 맞췄습니다)

     평가일       (a) 스냅샷 값       (b) 즉석 계산        차이   더 나은 쪽
--------------------------------------------------------------------
     142           0.886           0.896    +0.010   즉석
     145           0.876           0.916    +0.040   즉석
     148           0.913           0.920    +0.007   즉석
     151           0.906           0.920    +0.013   즉석
     154           0.923           0.923    +0.000   같음
     157           0.910           0.910    +0.000   같음
--------------------------------------------------------------------
      평균           0.902           0.914    +0.012

=> ⚠️ 즉석 계산이 한 번도 지지 않았습니다. 평균 +0.012.

   📗 원문은 '더 신선한 게 자동으로 더 낫진 않다' 고 조심스럽게 말합니다.
   측정해 보니 여기서는 신선한 쪽이 조금 낫거나 같았습니다.

   📝 그래서 이 절의 교훈은 '신선한 게 낫다' 가 아닙니다. 이겁니다.

      괴리 수치는 '두 쪽이 다르다' 만 말해 줍니다.
      '어느 쪽이 맞다' 는 안 말해 줍니다.

   불일치 78.6% 를 보고 '서빙이 틀렸다' 고 단정하면 안 됩니다.
   여기서는 오히려 서빙 쪽이 나았으니까요.
   괴리를 발견하면 어느 쪽이 맞는지는 정확도로 따로 

> **👀 주목할 곳** — `차이` 열. **음수가 하나도 없습니다.**
>
> **✅ 알게 된 사실** — ⚠️ 8절에서 불일치 78.6% 가 나왔는데, 정확도는 즉석 계산 쪽이 평균 +0.012 로 조금 **낫습니다.**
**괴리 수치는 '두 쪽이 다르다' 만 말하고 '어느 쪽이 맞다' 는 안 말합니다.**
②번 묵음은 설계 선택이고 ③번 코드 차이는 버그입니다. 같은 '괴리' 숫자라도 성격이 다릅니다.

> **🧩 이 절의 결론**
>
> ⚠️ 큰 괴리 수치가 **성능 손실을 뜻하지 않습니다.** 여기서는 오히려 서빙 쪽이 조금 나았습니다.
> 괴리 검사는 **일관성 검사**지 **품질 검사**가 아닙니다.
> 발견하면 어느 쪽이 맞는지는 **정확도로 따로** 재야 합니다.
>
> **➡️ 다음**
>
> 마지막으로 **설계 손잡이 두 개**를 돌려 봅니다.
> TTL 과 스냅샷 간격이요. 📗 원문의 연습문제 1·2번입니다.

---

## 10. ⚠️ TTL 과 스냅샷 간격

<sub>📗 원문 연습문제 1·2번 + ✨ 측정</sub>

피처 스토어를 세울 때 **직접 정해야 하는 숫자 두 개**입니다.

| 손잡이 | 조이면 | 풀면 |
| --- | --- | --- |
| **스냅샷 간격** | 신선해지지만 **백필 비용이 비례해서** 늘어남 | 싸지만 값이 묵음 |
| **TTL** | 묵은 값 대신 기본값을 더 자주 씀 | 묵은 값을 그대로 씀 |

흔한 직관은 **"스냅샷이 주간이면 TTL 도 7일쯤"** 인데,
재 보면 **그 둘이 생각만큼 연결돼 있지 않습니다.**

> **실험 10** · ⚠️ TTL 과 스냅샷 간격을 돌려 보기
>
> **🎯 배울 것** — 두 손잡이가 **실제로 무엇을 바꾸는지** 봅니다. 직관과 다른 데가 있습니다.
>
> **🔬 하는 일** — 📗 연습문제 2번: TTL 을 줄이며 특징을 잃는 학습 행을 셉니다. 연습문제 1번: 스냅샷 간격을 줄이며 괴리와 비용을 봅니다.

In [8]:
# 📗 연습문제 2번 — TTL 을 줄이면 기본값으로 때우는 행이 몇 개나
print("① TTL 을 줄이면 (스냅샷은 7일 간격 고정)\n")
print(f"{'TTL':>8}{'특징 있는 행':>14}{'기본값으로 때운 행':>20}{'비율':>9}")
print("-" * 54)
for ttl in (None, 10, 7, 5, 3, 2, 1):
    j = 시점정확조인(훈련라벨, 오프라인행, ttl)
    없음 = sum(1 for r in j if r[4] is None)
    print(f"{str(ttl):>8}{len(j) - 없음:>14}{없음:>20}{없음 / len(j):>9.1%}")

# 라벨일과 가장 가까운 과거 스냅샷 사이의 실제 간격
간격들 = []
for 엔, 라벨일, _ in 훈련라벨:
    이전 = [d for d in 스냅샷일자 if d <= 라벨일]
    if 이전:
        간격들.append(라벨일 - max(이전))
from collections import Counter
print(f"\n   라벨일과 직전 스냅샷의 실제 간격 분포: {dict(sorted(Counter(간격들).items()))}")
print(f"   최대 간격이 {max(간격들)}일 입니다.")
print(f"\n=> ⚠️ TTL 을 10에서 2까지 줄여도 아무것도 안 잃습니다.")
print(f"   '스냅샷이 7일 간격이니 TTL 7 미만은 위험하다' 는 직관이 빗나갑니다.")
print(f"\n   이유는 간격 분포에 있습니다. 라벨일이 100·120·140 인데")
print(f"   직전 스냅샷이 98·119·140 이라 간격이 각각 2·1·0 일뿐이거든요.")
print(f"   TTL 이 무는 건 스냅샷 간격이 아니라 라벨일과 스냅샷의 실제 간격입니다.")
print(f"\n   📝 그러니 TTL 을 정할 때는 스냅샷 주기를 보지 말고")
print(f"      내 라벨이 스냅샷 대비 어디에 떨어지는지 재 보세요.")

print("\n\n② 스냅샷 간격을 줄이면 (묵음 괴리 vs 백필 비용)\n")
print(f"{'간격':>7}{'백필 행':>10}{'비용':>8}   묵음 괴리(불일치 비율)")
print("-" * 68)
기준 = None
for 간격 in (7, 3, 1):
    일자 = list(range(간격, 176, 간격))
    행들2 = 백필(사건들, 일자)
    기준 = 기준 or len(행들2)
    j2 = 시점정확조인([(엔, 일, 0) for 엔, 일 in 요청들], 행들2, TTL일수)
    학습2 = {(r[0], r[1]): 등록소.기본값채우기(특징이름들, r[4]) for r in j2}
    보고 = 괴리검사(즉석값, 학습2, 특징이름들)
    괴리글 = "  ".join(f"{f.split('_')[0]} {비:>5.1%}" for f, _, 비, _, _ in 보고)
    print(f"{간격:>6}일{len(행들2):>10,}{len(행들2)/기준:>7.1f}배   {괴리글}")

print(f"\n=> 간격을 1일로 줄이면 묵음 괴리가 전부 0% 가 됩니다.")
print(f"   매일 돌리니 즉석 계산과 같아지는 거죠. 대신 비용이 7배입니다.")
print(f"\n   📝 8절 ② 의 '묵음 괴리' 는 버그가 아니라 이 비용을 아낀 대가입니다.")
print(f"      그리고 9절에서 봤듯 그 대가가 정확도로는 거의 안 나타났습니다.")
print(f"      즉, 7배를 더 쓸 이유가 여기선 없었습니다.")

① TTL 을 줄이면 (스냅샷은 7일 간격 고정)

     TTL       특징 있는 행          기본값으로 때운 행       비율
------------------------------------------------------
    None           897                   0     0.0%
      10           897                   0     0.0%
       7           897                   0     0.0%
       5           897                   0     0.0%
       3           897                   0     0.0%
       2           897                   0     0.0%
       1           598                 299    33.3%

   라벨일과 직전 스냅샷의 실제 간격 분포: {0: 299, 1: 299, 2: 299}
   최대 간격이 2일 입니다.

=> ⚠️ TTL 을 10에서 2까지 줄여도 아무것도 안 잃습니다.
   '스냅샷이 7일 간격이니 TTL 7 미만은 위험하다' 는 직관이 빗나갑니다.

   이유는 간격 분포에 있습니다. 라벨일이 100·120·140 인데
   직전 스냅샷이 98·119·140 이라 간격이 각각 2·1·0 일뿐이거든요.
   TTL 이 무는 건 스냅샷 간격이 아니라 라벨일과 스냅샷의 실제 간격입니다.

   📝 그러니 TTL 을 정할 때는 스냅샷 주기를 보지 말고
      내 라벨이 스냅샷 대비 어디에 떨어지는지 재 보세요.


② 스냅샷 간격을 줄이면 (묵음 괴리 vs 백필 비용)

     간격      백필 행      비용   묵음 괴리(불일치 비율)
----------------------------------------------------------------

> **👀 주목할 곳** — ① 표의 `비율` 열이 TTL 2 까지 **전부 0.0%** 라는 것, 그리고 그 아래 간격 분포.
>
> **✅ 알게 된 사실** — ⚠️ '스냅샷이 7일 간격이니 TTL 도 7일쯤' 이라는 직관이 빗나갑니다.
라벨일 100·120·140 의 직전 스냅샷이 98·119·140 이라 **실제 간격이 0~2일**뿐이거든요.
TTL 은 스냅샷 주기가 아니라 **라벨이 스냅샷 대비 어디 떨어지는지**에 물립니다.
그리고 ②에서 간격을 1일로 줄이면 괴리는 0이 되지만 **비용이 7배**입니다.

> **🧩 이 절의 결론**
>
> ⚠️ TTL 은 **스냅샷 주기가 아니라 라벨-스냅샷 실제 간격**에 물립니다. 직접 재 보고 정하세요.
> 스냅샷 간격은 **묵음 괴리와 백필 비용의 맞교환**입니다. 1일로 줄이면 괴리 0, 비용 7배.
> 9절에서 그 괴리가 정확도에 거의 영향이 없었으니, 여기선 7배를 쓸 이유가 없었습니다.
>
> **➡️ 다음**
>
> 11절에서 전체를 정리하고 Feast 대응표 · Ship It 스킬로 마칩니다.
> 이걸로 **19~21단원 묶음이 끝납니다.**

---

## 11. 정리

### ⚠️ 이 노트북에서 원문과 달랐거나, 원문이 한 줄로 넘어간 것

| 📗 원문 | 측정 결과 | 절 |
| --- | --- | --- |
| "더 신선한 값이 여기서 자동으로 더 나은 건 아니다" | ⚠️ 평가일 **6일 전부** 신선한 쪽이 같거나 나았습니다 (평균 +0.012). 괴리 78.6%가 정확도 손실로 이어지지 않았습니다 | 9 |
| 연습문제 2번 (TTL 을 3일로 줄여 보라) | ⚠️ TTL 을 **2일까지 줄여도 잃는 행이 0개**입니다. 라벨-스냅샷 실제 간격이 0~2일이라서요. 스냅샷 주기와는 별개입니다 | 10 |
| 연습문제 1번 (스냅샷을 1일로) | ✨ 묵음 괴리가 **전부 0%** 가 되고 백필 비용이 **7배**가 됩니다 | 10 |

**측정으로 확인된** 원문의 주장들입니다.

| 📗 원문의 주장 | 결과 | 절 |
| --- | --- | --- |
| 순진한 조인: 오프라인 0.948 / 실제 0.763 | ✅ 그대로 재현 | 3 |
| 시점 정확 조인: 오프라인 0.893 / 실제 0.876 | ✅ 그대로 재현 | 3 |
| 누수가 임곗값을 19일 → 53일로 민다 | ✅ 그대로 | 3 |
| 순진한 조인은 100% 행에 미래 스냅샷을 쓴다 | ✅ 100% | 3 |
| 온라인 스토어 vs 학습 조인은 괴리 0 | ✅ 세 특징 전부 0.0% | 8 |
| 옛 코드 경로는 창 개수에서 어긋난다 | ✅ 불일치 17.2% (원문 12.5%, 기록 표본이 달라서입니다) | 8 |
| 신규 사용자 기본값 −1 vs 365 | ✅ 불일치 6.8% 인데 평균 차이 24.85 | 8 |

### 🔑 한 줄로 꿰면

> **특징은 정의가 하나여야 합니다.**
> 학습은 **그 시점 값만** 읽고, 서빙은 **같은 정의로** 계산하고,
> 둘이 **진짜 같은지 계속 확인**합니다.

그리고 ⚠️ 하나 더.
**괴리 검사는 일관성 검사지 품질 검사가 아닙니다.**
"다르다" 와 "틀렸다" 는 다릅니다.

---

## 🔧 Feast 대응표

<sub>📗 원문 "Use It"</sub>

**Feast** 는 오픈소스 피처 스토어입니다. 이 노트북에서 만든 것들이 하나씩 대응됩니다.

| 이 노트북 | Feast |
| --- | --- |
| `특징정의`, `레지스트리` | 피처 레포의 `Entity` · `FeatureView` · `Field`, `feast apply` 로 등록 |
| `오프라인행` 이력 | 오프라인 스토어 — Parquet 위의 `FileSource` 나 웨어하우스 테이블 |
| `시점정확조인` + TTL | `store.get_historical_features(entity_df=..., features=[...])` |
| `머티리얼라이즈` | `feast materialize-incremental <끝 시각>` |
| `온라인스토어.읽기` | `store.get_online_features(features=[...], entity_rows=[...])` |
| 생성 시각 확인 (2절) | 소스의 `created_timestamp_column` + `filter_by_created_timestamp=True` |

```python
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

user = Entity(name="user", join_keys=["user_id"])

user_activity_source = FileSource(
    name="user_activity_source",
    path="data/user_activity.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",      # 2절의 '생성 시각'
)

user_activity = FeatureView(
    name="user_activity",
    entities=[user],
    ttl=timedelta(days=10),                  # 7절의 TTL
    schema=[
        Field(name="purchases_30d", dtype=Int64),
        Field(name="spend_30d", dtype=Float32),
        Field(name="days_since_last_purchase", dtype=Int64),
    ],
    source=user_activity_source,
)
```

그리고 **학습과 서빙이 같은 피처 뷰를 읽습니다.**

```python
training_df = store.get_historical_features(
    entity_df=labels_df,                          # user_id, event_timestamp, label
    features=["user_activity:purchases_30d",
              "user_activity:days_since_last_purchase"],
    filter_by_created_timestamp=True,             # 백필 누수 차단 (2절)
).to_df()

online = store.get_online_features(
    features=["user_activity:purchases_30d",
              "user_activity:days_since_last_purchase"],
    entity_rows=[{"user_id": 42}],
).to_dict()
```

> 📗 Feast 문서가 명시하길, **TTL 은 엔티티 데이터프레임의 각 타임스탬프 기준**이지
> **쿼리 시점 기준이 아닙니다.** 이 노트북 `시점정확조인` 의 TTL 검사와 같습니다.
>
> 그리고 `filter_by_created_timestamp=True` 가 **백필 값이 학습 데이터로 새는 걸** 막습니다.
> 모든 오프라인 스토어가 이 옵션을 지원하진 않는데,
> **지원 안 하는 스토어는 조용히 무시하지 않고 에러를 냅니다.**

Tecton · Hopsworks · 대형 클라우드 ML 플랫폼의 피처 스토어도 **같은 오프라인/온라인 분리**를 씁니다.
괴리 검사는 **TensorFlow Data Validation** 이 학습 데이터와 서빙 데이터를 비교해 줍니다.

---

## 🚀 Ship It — `skill-feature-skew-audit.md`

<sub>📗 원문 저장소의 `21-feature-stores-and-training-serving-skew/outputs/skill-feature-skew-audit.md` (76줄)</sub>

> 📗 원문: 이 스킬은 특징 파이프라인에서 **시간 여행과 학습-서빙 괴리**를 감사합니다.
> 조인 · 정의 · 기본값 · 각 경로의 신선도, 그리고 괴리 검사에 필요한 **기록**을 차례로 봅니다.

### 감사 순서

| 순서 | 묻는 것 | 통과 기준 |
| --- | --- | --- |
| 1 | **조인이 시점 정확한가** | 라벨 시각 이하 행만 쓰는가 (3절) |
| 2 | **정의가 한 곳에 있는가** | 서빙이 변환 로직 사본을 갖고 있지 않은가 (5절) |
| 3 | **기본값이 같은가** | 양쪽이 레지스트리에서 읽는가 (7절) |
| 4 | **각 경로의 신선도** | 서빙이 읽는 값의 나이가 학습 때와 같은가 (8절 ②) |
| 5 | **서빙 특징을 기록하는가** | 기록이 없으면 괴리 검사를 아예 못 합니다 (8절) |

### ✨ 추가 — 측정 뒤에 제가 덧붙인 세 질문

```markdown
## 괴리 수치를 받았을 때 (⚠️ 9절)

"불일치 78.6%" 를 보고 바로 고치러 가지 마세요.
괴리 검사는 두 쪽이 **다르다**는 것만 말하고
어느 쪽이 **맞다**는 건 안 말합니다.

측정해 보니 불일치 78.6% 인 경로가 정확도는 오히려 조금 나았습니다.
먼저 나누세요.
- 의도된 차이(묵음)인가, 의도된 적 없는 차이(창 하나, 기본값 -1)인가
- 의도된 차이라면 **정확도로 따로 재서** 어느 쪽이 나은지 확인

## TTL 을 정할 때 (⚠️ 10절)

"스냅샷이 주간이니 TTL 도 7일" 로 정하지 마세요.
TTL 이 무는 건 스냅샷 주기가 아니라
**라벨일과 직전 스냅샷 사이의 실제 간격**입니다.

그 간격 분포를 먼저 찍어 보세요.
이 노트북에서는 최대 2일이라 TTL 2 까지 내려도 아무것도 안 잃었습니다.

## 스냅샷 간격을 줄이자는 제안을 받았을 때 (10절)

비용이 간격에 **반비례**합니다 (7일 -> 1일이면 7배).
그 돈을 쓰기 전에 9절처럼 **정확도 차이를 먼저 재세요.**
괴리가 0 이 되는 건 맞지만, 그게 성능으로 돌아오는지는 별개입니다.
```

---

## 📚 용어 정리

<sub>📗 원문 "Key Terms" — 설명은 쉬운 말로 다시 썼습니다</sub>

| 용어 | 흔히 하는 말 | 진짜 무슨 뜻인가 |
| --- | --- | --- |
| **피처 스토어** | "특징 데이터베이스" | **정의 한 벌**에서 특징을 계산해 **학습과 서빙 양쪽에** 제공하는 시스템 |
| **오프라인 스토어** | "학습 데이터" | 이벤트 시각이 붙은 **전체 이력**. 학습 작업과 백필이 읽습니다 |
| **온라인 스토어** | "특징 캐시" | 엔티티당 **최신 값 하나**. 서빙 경로가 밀리초 단위로 읽습니다 |
| **엔티티** | "아이디" | 그 특징이 설명하는 대상의 키 |
| **이벤트 시각** | "언제 일어났나" | 그 값이 **참이 된** 시점 |
| **생성 시각** | — | 시스템이 **그 행을 쓴** 시점. 백필이면 둘이 다릅니다 |
| **시점 정확 조인** | "as-of 조인" | 각 라벨에 **그 시각 이하의 가장 최근** 특징 행을 붙이는 조인 |
| **TTL** | "만료" | 읽는 쪽이 **없는 셈 치기 시작하는** 값의 나이 |
| **머티리얼라이제이션** | "온라인 동기화" | 오프라인 스토어의 최신 값을 온라인 스토어로 **복사하는 작업** |
| **백필** | "과거 계산하기" | 특징 정의를 **과거 날짜마다 실행**해 이력을 만드는 것 |
| **학습-서빙 괴리** | "오프라인에선 됐는데" | 학습 때와 서빙 때의 **특징이나 동작이 다른 것** |

---

## 📖 더 읽을 것

<sub>📗 원문 "Further Reading"</sub>

- [Feast, Point-in-time joins](https://docs.feast.dev/getting-started/concepts/point-in-time-joins) — 3절의 TTL 의미와 생성 시각 필터
- [Feast 문서](https://docs.feast.dev/) — 피처 뷰, 오프라인/온라인 스토어, 머티리얼라이제이션
- [Google, *Rules of Machine Learning*](https://developers.google.com/machine-learning/guides/rules-of-ml) — 괴리 관련 규칙들(29번~37번). 29번은 "서빙 특징을 기록하라", 32번은 "코드를 재사용하라"
- [Breck et al., *Data Validation for Machine Learning* (SysML 2019)](https://research.google/pubs/data-validation-for-machine-learning/) — 구글의 데이터 검증 시스템과 학습-서빙 괴리
- [TensorFlow Data Validation 가이드](https://www.tensorflow.org/tfx/guide/tfdv) — 스키마 검사, 괴리 탐지, 드리프트 탐지
- [Sculley et al., *Hidden Technical Debt in ML Systems* (NeurIPS 2015)](https://papers.nips.cc/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html) — 19단원 7절에서도 본 그 논문. 데이터 의존성과 파이프라인 부채

---

## ✅ 이해 확인 퀴즈

<sub>📗 원문 저장소의 `21-feature-stores-and-training-serving-skew/quiz.json` + ✨ 측정 결과를 묻는 문항</sub>

**1. 오프라인 정확도 0.948, 실서비스 0.763 입니다. 가장 먼저 의심할 것은?**
1. 모델이 과적합됐다
2. **학습 데이터에 라벨 이후 값이 섞였다 (시간 여행)**
3. 서빙 지연이 너무 길다
4. 특징이 부족하다

**2. 시점 정확 조인이 16일 라벨에 붙이는 행은?**
1. 28일 행 (가장 최신)
2. 7일 행 (가장 오래됨)
3. **14일 행 (16일 이하 중 가장 최근)**
4. 21일 행

**3. 괴리 검사에서 ① 온라인 vs 학습 조인이 0%, ③ 옛 코드 vs 즉석 계산이 17.2% 입니다. 범인은?**
1. 온라인 스토어가 고장났다
2. **옛 코드 경로. 같은 시점인데 코드만 다르니 코드 차이가 원인**
3. 묵은 특징
4. TTL 설정

**4. ⚠️ 불일치 78.6% 가 나왔습니다. 바로 고쳐야 할까요?**
1. 예. 78.6% 는 명백한 버그입니다
2. **어느 쪽이 맞는지 정확도로 따로 재야 합니다. 측정에선 그 경로가 오히려 나았습니다**
3. 아니오. 괴리는 무시해도 됩니다
4. TTL 을 늘리면 해결됩니다

**5. ⚠️ 스냅샷이 7일 간격입니다. TTL 은 몇으로?**
1. 7일. 스냅샷 간격과 맞춰야 합니다
2. 14일. 여유를 둬야 합니다
3. **라벨일과 직전 스냅샷의 실제 간격 분포를 찍어 보고 정합니다**
4. TTL 은 쓰지 않습니다

In [9]:
문항들 = [
    ("오프라인은 좋고 실제는 나쁨", 2,
     "측정에서 순진한 조인은 오프라인 0.948 / 실제 0.763, 시점 정확 조인은 0.893 / 0.876 이었습니다. "
     "순진한 조인이 라벨 100% 에 미래 스냅샷을 붙였고, 그게 임곗값을 19일에서 53일로 밀어 올렸습니다. "
     "누수는 더 나쁜 모델에 더 좋은 점수를 줍니다.",
     "3절 (두 조인을 같은 평가 데이터로 비교)"),
    ("시점 정확 조인이 고르는 행", 3,
     "라벨 시각 이하 중 가장 최근을 고릅니다. 21일과 28일 행은 미래라 버립니다. "
     "28일 행은 이미 라벨 기간이 끝난 뒤를 세고 있어서, 쓰면 정답이 입력에 복사됩니다.",
     "3절 (네 단계와 표)"),
    ("세 쌍 비교로 범인 찾기", 2,
     "각 비교가 변수를 하나씩만 다르게 둡니다. ①은 정의도 스냅샷도 같으니 0%가 정상이고, "
     "③은 시점이 같은데 코드만 다르니 차이가 전부 코드 탓입니다. "
     "창에 하루가 더 들어가서 17.2% 가 어긋났습니다.",
     "8절 (세 쌍 비교)"),
    ("불일치 78.6% 를 받았을 때", 2,
     "그 경로(즉석 계산)로 평가해 보니 6개 평가일 전부 같거나 나았고 평균 +0.012 였습니다. "
     "괴리 검사는 '두 쪽이 다르다' 만 말하고 '어느 쪽이 맞다' 는 안 말합니다. "
     "다만 8절 ③번 같은 코드 차이는 의도된 적이 없으니 버그가 맞습니다.",
     "9절 (스냅샷 값 vs 즉석 계산 정확도 비교)"),
    ("TTL 을 정하는 법", 3,
     "측정에서 TTL 을 2일까지 줄여도 잃는 행이 0개였습니다. "
     "라벨일 100·120·140 의 직전 스냅샷이 98·119·140 이라 실제 간격이 2·1·0 일뿐이었거든요. "
     "TTL 은 스냅샷 주기가 아니라 라벨-스냅샷 실제 간격에 물립니다.",
     "10절 (TTL 을 줄이며 간격 분포 확인)"),
]

print("=" * 80)
for 번호, (제목, 정답, 해설, 어디) in enumerate(문항들, 1):
    print(f"{번호}. {제목}")
    print(f"   정답: {정답}번")
    print(f"   해설: {해설}")
    print(f"   이 노트북: {어디}")
    print("-" * 80)

print("\n채점 기준")
print("   5개 다 맞음 -> 19~21단원 묶음을 다 소화하셨습니다")
print("   3~4개 맞음  -> 틀린 문항의 '이 노트북' 절만 다시 읽으세요")
print("   2개 이하    -> 3절(시점 정확 조인)과 8절(세 쌍 비교)을 먼저 보세요")

print("\n\n19~21단원 묶음 정리")
print("   19단원  설계    — 배포 전에 정할 것 (결정 -> 신선도 -> 서빙 -> 비용)")
print("   20단원  감시    — 세상이 변할 때 (입력은 빠르지만 눈이 멀고, 결과는 정확하지만 늦다)")
print("   21단원  일관성  — 코드가 두 벌일 때 (정의는 하나, 독자는 둘)   <- 지금 여기")
print("\n   세 단원 다 '모델 바깥' 이야기였습니다. 1~18단원이 모델 안쪽이었다면요.")
print("\n   그리고 세 단원이 같은 교훈을 다르게 말합니다.")
print("   오프라인 숫자 하나만 보고 판단하지 마세요.")
print("   19단원은 기준선과 비교하라고, 20단원은 계속 감시하라고,")
print("   21단원은 학습 데이터 자체를 의심하라고 말합니다.")

1. 오프라인은 좋고 실제는 나쁨
   정답: 2번
   해설: 측정에서 순진한 조인은 오프라인 0.948 / 실제 0.763, 시점 정확 조인은 0.893 / 0.876 이었습니다. 순진한 조인이 라벨 100% 에 미래 스냅샷을 붙였고, 그게 임곗값을 19일에서 53일로 밀어 올렸습니다. 누수는 더 나쁜 모델에 더 좋은 점수를 줍니다.
   이 노트북: 3절 (두 조인을 같은 평가 데이터로 비교)
--------------------------------------------------------------------------------
2. 시점 정확 조인이 고르는 행
   정답: 3번
   해설: 라벨 시각 이하 중 가장 최근을 고릅니다. 21일과 28일 행은 미래라 버립니다. 28일 행은 이미 라벨 기간이 끝난 뒤를 세고 있어서, 쓰면 정답이 입력에 복사됩니다.
   이 노트북: 3절 (네 단계와 표)
--------------------------------------------------------------------------------
3. 세 쌍 비교로 범인 찾기
   정답: 2번
   해설: 각 비교가 변수를 하나씩만 다르게 둡니다. ①은 정의도 스냅샷도 같으니 0%가 정상이고, ③은 시점이 같은데 코드만 다르니 차이가 전부 코드 탓입니다. 창에 하루가 더 들어가서 17.2% 가 어긋났습니다.
   이 노트북: 8절 (세 쌍 비교)
--------------------------------------------------------------------------------
4. 불일치 78.6% 를 받았을 때
   정답: 2번
   해설: 그 경로(즉석 계산)로 평가해 보니 6개 평가일 전부 같거나 나았고 평균 +0.012 였습니다. 괴리 검사는 '두 쪽이 다르다' 만 말하고 '어느 쪽이 맞다' 는 안 말합니다. 다만 8절 ③번 같은 코드 차이는 의도된 적이 없으니 버그가 맞습니다.
   이 노트북: 9절 (스냅샷 값 vs 즉석 계산 정